# 🎓 Προσομοίωση Μετακινήσεων & Περιβαλλοντικού Αποτυπώματος Φοιτητών (ΠΑΔΑ)

Αυτό το Jupyter Notebook αναλύει βήμα-βήμα τον κώδικα προσομοίωσης `simulate_failing_students_co2.py`.
Ο κώδικας έχει χωριστεί σε θεματικές ενότητες με εκτενή επεξήγηση στα Ελληνικά για κάθε τμήμα του.

## 📍 Section 1: Εισαγωγή Βιβλιοθηκών & Αρχική Παραμετροποίηση

Σε αυτό το πρώτο κελί, εισάγουμε τις απαραίτητες βιβλιοθήκες της Python:
* `csv` και `json` για την ανάγνωση των δεδομένων των φοιτητών και των ταχυδρομικών κωδικών.
* `math` για τις τριγωνομετρικές συναρτήσεις του υπολογισμού αποστάσεων (Haversine) και τον εκθετικό υπολογισμό του Logit.
* `random` για την Monte Carlo προσομοίωση της τελικής επιλογής.
* `os` για τη διαχείριση των μονοπατιών των αρχείων.
* `requests` για την επικοινωνία με τους τοπικούς OSRM servers (Docker containers).

Επίσης, ορίζουμε τις σταθερές εισόδου:
* `DEST_LAT` & `DEST_LON`: Οι γεωγραφικές συντεταγμένες του UNIWA Campus 2 (Αρχαίος Ελαιώνας).
* `EF_...`: Οι συντελεστές εκπομπών CO2 ανά επιβάτη ανά χιλιόμετρο (g CO2eq / pax / km).
* `ASC_...`: Οι σταθερές προτίμησης μέσου (Alternative Specific Constants).
* `THETA`: Η παράμετρος ευαισθησίας κόστους του μοντέλου Logit.

In [ ]:
import csv
import json
import math
import random
import os
import requests

# Ορισμός φακέλου εργασίας και μονοπατιών αρχείων δεδομένων
BASE_DIR = os.path.dirname(os.path.abspath(''))
POSTCODES_PATH = os.path.join(BASE_DIR, 'data', 'postcodes_attica.json')
DATASET_PATH = os.path.join(BASE_DIR, 'data', 'students_exam_dataset.csv')

# Φόρτωση της βάσης δεδομένων με τις συντεταγμένες των Τ.Κ. Αττικής
with open(POSTCODES_PATH, 'r', encoding='utf-8') as f:
    local_postcodes = json.load(f)


# Destination campus coordinates map
UNIVERSITIES = {
    "UNIWA Egaleo": (37.9857, 23.6792),
    "EKPA Zografou": (37.9676, 23.7665),
    "EKPA Goudi": (37.9834, 23.7681),
    "EKPA Center": (37.9804, 23.7335),
    "EKPA Dafni": (37.9546, 23.7431),
    "EMP Zografou": (37.9760, 23.7840),
    "EMP Patision": (37.9877, 23.7313),
    "OPA Center": (37.9942, 23.7328),
    "OPA Evelpidon": (37.9961, 23.7381),
    "OPA Troias": (37.9985, 23.7345),
    "PADA Egaleo": (38.0031, 23.6758),
    "PADA Ralli": (37.9818, 23.6765),
    "PADA Alexandras": (37.9880, 23.7550),
    "PAPEI Center": (37.9416, 23.6528),
    "PAPEI Lampraki": (37.9415, 23.6552),
    "Panteion": (37.9602, 23.7196),
    "GPA Votanikos": (37.9827, 23.7051),
    "Harokopio": (37.9610, 23.7088),
    "ASKT Tavros": (37.9617, 23.6888),
    "Ikaron": (38.1091, 23.7828),
    "N. Dokimon": (37.9282, 23.6288),
    "Evelpidon": (37.8282, 23.7744)
}

TARGET_CAMPUS = "UNIWA Egaleo" # Change this to run for other campuses!
DEST_LAT, DEST_LON = UNIVERSITIES.get(TARGET_CAMPUS, (37.9857, 23.6792))


# Συντελεστές εκπομπών CO2 ανά μέσο (g CO2eq / επιβάτη / km)
EF_CAR   = 120.0
EF_MOTO  = 70.0
EF_BUS   = 10.81  # Σταθμισμένος μέσος όρος στόλου ΟΣΥ (Diesel, CNG, Electric)
EF_METRO = 3.1    # Ηλεκτροκίνητο μέσο σταθερής τροχιάς
EF_FOOT  = 0.0    # Μηδενικές εκπομπές για πεζή μετακίνηση

# Σταθερές προτίμησης μέσου (Alternative-Specific Constants - ASC)
# Θετική τιμή = Ποινή (μειώνει την ελκυστικότητα του μέσου)
# Αρνητική τιμή = Bonus (αυξάνει την ελκυστικότητα του μέσου)
ASC_CAR   = 12.0  # Αντιπροσωπεύει το άγχος οδήγησης, συντήρησης και ασφάλειας
ASC_MOTO  = 14.0  # Υψηλότερος κίνδυνος ατυχήματος και έκθεση στις καιρικές συνθήκες
ASC_T1    = -8.0  # Bonus για Metro + Bus λόγω φοιτητικής έκπτωσης και άνεσης
ASC_T2    = -6.0  # Bonus για απευθείας Λεωφορείο
ASC_FOOT  = 0.0   # Baseline μέσο αναφοράς
THETA     = 0.09  # Παράμετρος ευαισθησίας κόστους

## 📍 Section 2: Ντετερμινιστική Διαθεσιμότητα Οχημάτων (`pseudo_hash`)

Για να προσομοιώσουμε αν ένας φοιτητής έχει πρόσβαση σε αυτοκίνητο (20% πιθανότητα) ή μηχανή (15% πιθανότητα) χωρίς να αλλάζουν τα αποτελέσματα σε κάθε εκτέλεση του κώδικα, χρησιμοποιούμε έναν ντετερμινιστικό αλγόριθμο κατακερματισμού (hash).
Παίρνει ως είσοδο τον Τ.Κ. και επιστρέφει έναν σταθερό ακέραιο αριθμό.

In [ ]:
def pseudo_hash(s):
    # Παραγωγή σταθερού ακεραίου αριθμού από το string του Τ.Κ. για επαναληψιμότητα
    h = 0
    for char in s:
        h = (h * 31 + ord(char)) & 0xFFFFFFFF
    return h

## 📍 Section 3: Επικοινωνία με τους Docker OSRM Containers (`fetch_osrm_route`)

Η συνάρτηση αυτή πραγματοποιεί HTTP requests στους τοπικούς servers του OSRM (Open Source Routing Machine) που τρέχουν σε Docker containers:
* `Port 5000`: OSRM Car Engine
* `Port 5001`: OSRM Walking Engine

Αν οι OSRM servers είναι offline, επιστρέφει `None` και το σύστημα μεταβαίνει σε fallback υπολογισμό (Haversine).

In [ ]:
def fetch_osrm_route(port, profile, lat1, lon1, lat2, lon2):
    # Κατασκευή URL κλήσης στο τοπικό OSRM Docker instance
    url = f"http://localhost:{port}/route/v1/{profile}/{lon1},{lat1};{lon2},{lat2}"
    try:
        # Αίτημα HTTP GET με timeout 1.5 δευτερόλεπτο για αποφυγή «παγώματος»
        res = requests.get(url, params={"overview": "false"}, timeout=1.5)
        if res.status_code == 200:
            data = res.json()
            if data.get('code') == 'Ok' and data.get('routes'):
                r = data['routes'][0]
                return {
                    'dist_km': r['distance'] / 1000.0, # Μετατροπή μέτρων σε χιλιόμετρα
                    'dur_min': r['duration'] / 60.0    # Μετατροπή δευτερολέπτων σε λεπτά
                }
    except Exception:
        pass
    return None

## 📍 Section 4: Μαθηματικός Υπολογισμός Γεωδαιτικής Απόστασης (Haversine Formula)

Η συνάρτηση haversine_km είναι θεμελιώδης για το μοντέλο μας και εξυπηρετεί δύο κρίσιμους ρόλους:

1. **Μοντελοποίηση Μέσων Μαζικής Μεταφοράς (Λεωφορείο & Μετρό)**:
   Επειδή δεν έχουμε φορτώσει GTFS δεδομένα δρομολογίων ΜΜΜ σε OSRM engine, οι αποστάσεις και οι χρόνοι των ΜΜΜ υπολογίζονται μακροσκοπικά.
   Χρησιμοποιούμε τη γεωδαιτική απόσταση (ευθεία) της Haversine ως βάση, και εφαρμόζουμε διεθνώς αναγνωρισμένους συγκοινωνιακούς συντελεστές, όπως ο **Συντελεστής Στρεβλότητας (Circuity Factor = 1.15)** για την προσέγγιση της πραγματικής απόστασης των λεωφορειακών γραμμών.

2. **Εφεδρεία (Fallback)**:
   Αν ο τοπικός OSRM server (Docker) παρουσιάσει στιγμιαία καθυστέρηση ή βγει offline, η συνάρτηση αυτή εξασφαλίζει ότι η προσομοίωση θα συνεχίσει να εκτελείται ομαλά χωρίς σφάλματα (crash), χρησιμοποιώντας μια γεωμετρική προσέγγιση (π.χ. απόσταση Ι.Χ. = ευθεία * 1.25).


In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    # Μαθηματικός τύπος Haversine για την απόσταση δύο γεωγραφικών σημείων
    R = 6371.0 # Ακτίνα της Γης σε χιλιόμετρα
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2)**2
    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))

## 📍 Section 5: Προσομοίωση Μετακίνησης Φοιτητή (`compute_student_route_and_co2`)

Αυτή η συνάρτηση εκτελεί το σύνολο των υπολογισμών για έναν μεμονωμένο φοιτητή:
1. **Έλεγχος Τ.Κ.**: Επαληθεύει αν ο Τ.Κ. υπάρχει στη βάση.
2. **Διαθεσιμότητα Οχημάτων**: Ελέγχει αν ο φοιτητής ανήκει στο ποσοστό που διαθέτει Ι.Χ. ή Μηχανή (1ο Φίλτρο).
3. **Λήψη Γεωμετρίας & Χρόνων**: Καλεί το OSRM για να βρει αποστάσεις/χρόνους.
4. **Μοντελοποίηση ΜΜΜ**: Προσομοιώνει δύο σενάρια ΜΜΜ (Μετρό + Λεωφορείο έναντι Απευθείας Λεωφορείου) με βάση την ώρα (Peak / Off-Peak).
5. **Γενικευμένο Κόστος ($C_i$)**: Υπολογίζει το υποκειμενικό κόστος κάθε μέσου (με ποινή πάρκινγκ μόνο στο «Πήγαινε» και αυξημένο κόστος καυσίμων).
6. **Μοντέλο Logit**: Υπολογίζει τις πιθανότητες επιλογής.
7. **Monte Carlo Lottery**: Επιλέγει τυχαία (αλλά βάσει των πιθανοτήτων) το επικρατέστερο μέσο.

In [ ]:
def compute_student_route_and_co2(tk, is_peak=True, reverse=False):
    clean_tk = str(tk).strip()
    if clean_tk not in local_postcodes:
        return None

    coord = local_postcodes[clean_tk]
    lat, lon = coord['lat'], coord['lon']

    # 1ο Φίλτρο: Έλεγχος αν ο φοιτητής διαθέτει Ι.Χ. (20%) ή Μηχανή (15%)
    h = pseudo_hash(clean_tk)
    has_car = (h % 100 < 20)
    has_moto = ((h >> 2) % 100 < 15)

    # Λήψη δεδομένων δρομολόγησης (ανάλογα με την κατεύθυνση)
    if not reverse:
        osrm_car  = fetch_osrm_route(5000, "driving", lat, lon, DEST_LAT, DEST_LON)
        osrm_foot = fetch_osrm_route(5001, "walking", lat, lon, DEST_LAT, DEST_LON)
    else:
        osrm_car  = fetch_osrm_route(5000, "driving", DEST_LAT, DEST_LON, lat, lon)
        osrm_foot = fetch_osrm_route(5001, "walking", DEST_LAT, DEST_LON, lat, lon)

    # Υπολογισμός αποστάσεων/χρόνων με fallback σε Haversine αν OSRM είναι offline
    straight_km = haversine_km(lat, lon, DEST_LAT, DEST_LON)
    car_dist_km  = osrm_car['dist_km'] if osrm_car else (straight_km * 1.25)
    car_dur_min  = osrm_car['dur_min'] if osrm_car else max(4.0, car_dist_km * 2.8)
    
    foot_dist_km = osrm_foot['dist_km'] if osrm_foot else (straight_km * 1.15)
    foot_dur_min = osrm_foot['dur_min'] if osrm_foot else max(3.0, foot_dist_km * 13.0)

    bus_dist_km  = straight_km * 1.15
    moto_dist_km = car_dist_km
    moto_dur_min = car_dur_min * 0.90

    # Χρόνοι αναμονής ΜΜΜ (μικρότεροι στις ώρες αιχμής λόγω συχνότητας δρομολογίων)
    metro_wait = 4.0 if is_peak else 6.0
    bus_wait   = 7.0 if is_peak else 11.0

    # Transit Option 1: Metro + Bus (Συνδυαστικό)
    t1_in_metro  = max(3.0, straight_km * 1.5)
    t1_in_bus    = max(4.0, straight_km * 1.8)
    t1_wait      = metro_wait + bus_wait
    t1_walk      = 7.0
    t1_total_dur = t1_walk + t1_wait + t1_in_metro + t1_in_bus
    t1_co2_grams = round((bus_dist_km * 0.45 * EF_METRO) + (bus_dist_km * 0.55 * EF_BUS))

    # Transit Option 2: Direct Bus (Απευθείας Λεωφορείο)
    t2_in_bus    = max(8.0, straight_km * 3.1)
    t2_wait      = 9.0 if is_peak else 14.0
    t2_walk      = 9.0
    t2_total_dur = t2_walk + t2_wait + t2_in_bus
    t2_co2_grams = round(bus_dist_km * EF_BUS)

    # CO2 εκπομπές για Ι.Χ., Μηχανή και Περπάτημα
    car_co2_grams  = round(car_dist_km * EF_CAR)
    moto_co2_grams = round(car_dist_km * EF_MOTO)
    foot_co2_grams = 0.0

    # 2ο Φίλτρο: Ποινή αναζήτησης πάρκινγκ (εφαρμόζεται ΜΟΝΟ κατά τη μετάβαση στη σχολή)
    parking_car  = 4.0 if not reverse else 0.0
    parking_moto = 1.0 if not reverse else 0.0
    
    # Υπολογισμός Γενικευμένου Κόστους (Generalized Cost)
    # Περιλαμβάνει χρόνο, οικονομικό κόστος καυσίμων (0.35/km για Ι.Χ.) και το ASC bias
    C_car  = (car_dur_min + parking_car) + (car_dist_km * 0.35 * 5) + ASC_CAR
    C_moto = (moto_dur_min + parking_moto) + (car_dist_km * 0.18 * 5) + ASC_MOTO
    C_t1   = (t1_in_metro + t1_in_bus + t1_walk) + (1.2 * t1_wait) + 4.0 + ASC_T1
    C_t2   = (t2_in_bus + t2_walk) + (1.2 * t2_wait) + ASC_T2
    C_foot = foot_dur_min * 1.1 + ASC_FOOT

    # Διορθωτικό bias απόστασης για κοντινές πεζές μετακινήσεις
    if foot_dist_km <= 0.6:
        foot_bias = -30.0
    elif foot_dist_km <= 1.0:
        foot_bias = -12.0
    elif foot_dist_km <= 1.5:
        foot_bias = 0.0
    else:
        foot_bias = 30.0

    C_foot_adj = C_foot + foot_bias

    # Υπολογισμός εκθετικών όρων Logit (μηδενισμός αν δεν υπάρχει διαθέσιμο όχημα)
    exp_car  = math.exp(-THETA * C_car) if has_car else 0.0
    exp_moto = math.exp(-THETA * C_moto) if has_moto else 0.0
    exp_t1   = math.exp(-THETA * C_t1)
    exp_t2   = math.exp(-THETA * C_t2)
    exp_foot = math.exp(-THETA * C_foot_adj)
    sum_exp  = exp_car + exp_moto + exp_t1 + exp_t2 + exp_foot

    # Πιθανότητες Logit P(i)
    p_car  = exp_car / sum_exp
    p_moto = exp_moto / sum_exp
    p_t1   = exp_t1 / sum_exp
    p_t2   = exp_t2 / sum_exp
    p_foot = exp_foot / sum_exp

    # Κανονικοποίηση πιθανοτήτων ώστε το άθροισμά τους να ισούται ακριβώς με 1.0
    total_p = p_car + p_moto + p_t1 + p_t2 + p_foot
    p_car  /= total_p
    p_moto /= total_p
    p_t1   /= total_p
    p_t2   /= total_p
    p_foot /= total_p

    # Monte Carlo Lottery (Κλήρωση βάσει πιθανοτήτων)
    modes = [
        ('transit1', 'Metro + Express Bus', p_t1, t1_co2_grams, t1_total_dur),
        ('transit2', 'Direct Bus',          p_t2, t2_co2_grams, t2_total_dur),
        ('car',      'Car',                 p_car, car_co2_grams, car_dur_min),
        ('moto',     'Motorcycle',          p_moto, moto_co2_grams, moto_dur_min),
        ('foot',     'Walking',             p_foot, foot_co2_grams, foot_dur_min)
    ]

    r = random.random()
    cumulative = 0.0
    chosen_mode = modes[0]

    for mode in modes:
        cumulative += mode[2]
        if r <= cumulative:
            chosen_mode = mode
            break

    return {
        'tk': clean_tk,
        'dist_km': round(car_dist_km, 2),
        'probabilities': {m[0]: round(m[2]*100, 1) for m in modes},
        'chosen_mode_id': chosen_mode[0],
        'chosen_mode_name': chosen_mode[1],
        'chosen_co2_grams': chosen_mode[3],
        'chosen_dur_min': round(chosen_mode[4], 1)
    }

## 📍 Section 6: Εκτέλεση της Συνολικής Προσομοίωσης (`run_simulation`)

Αυτό το κελί εκτελεί την πλήρη προσομοίωση για τους φοιτητές που φιλτράρονται βάσει βαθμού:
1. Διαβάζει το αρχείο CSV.
2. Φιλτράρει τους φοιτητές με βαθμό `0.0` έως `1.0` (Failing students).
3. Για κάθε φοιτητή, τρέχει τη μετάβαση (Outbound, Ώρες Αιχμής) και την επιστροφή (Inbound, Ώρες Μη Αιχμής).
4. Αθροίζει τις CO2 εκπομπές και υπολογίζει τη διασπορά των μέσων (Mode Split).
5. Εκτυπώνει τη συνολική αναφορά.

In [ ]:
def run_simulation(min_grade=0.0, max_grade=1.0, dataset_path=DATASET_PATH):
    filtered_students = []
    invalid_count = 0

    # Ανάγνωση δεδομένων φοιτητών από το CSV
    with open(dataset_path, 'r', encoding='utf-8-sig') as f:
        reader = csv.DictReader(f)
        for row in reader:
            grade_str = row['GRADE'].strip()
            tk = row['TK_KATOIKIA'].strip()

            try:
                grade_val = float(grade_str)
                if min_grade <= grade_val <= max_grade:
                    filtered_students.append({'tk': tk, 'grade': grade_val, 'course': row['COURSE']})
            except ValueError:
                continue

    print(f"============================================================")
    print(f"Filters: Found {len(filtered_students)} students with grade {min_grade} to {max_grade}")
    print(f"============================================================\n")

    total_co2_grams = 0.0
    mode_counts = {'transit1': 0, 'transit2': 0, 'car': 0, 'moto': 0, 'foot': 0}
    mode_co2    = {'transit1': 0, 'transit2': 0, 'car': 0, 'moto': 0, 'foot': 0}
    valid_simulated = 0

    for i, s in enumerate(filtered_students, 1):
        # 1. Προσομοίωση Μετάβασης (Outbound - Peak - reverse=False)
        go_res = compute_student_route_and_co2(s['tk'], is_peak=True, reverse=False)
        # 2. Προσομοίωση Επιστροφής (Inbound - Off Peak - reverse=True)
        ret_res = compute_student_route_and_co2(s['tk'], is_peak=False, reverse=True)

        if not go_res or not ret_res:
            invalid_count += 1
            print(f"Student {i:02d} (Postcode {s['tk']}) excluded: invalid postcode or out of Attica")
            continue

        valid_simulated += 1
        
        # Στατιστικά μετάβασης
        go_id = go_res['chosen_mode_id']
        go_name = go_res['chosen_mode_name']
        go_co2 = go_res['chosen_co2_grams']
        go_dur = go_res['chosen_dur_min']

        # Στατιστικά επιστροφής
        ret_id = ret_res['chosen_mode_id']
        ret_name = ret_res['chosen_mode_name']
        ret_co2 = ret_res['chosen_co2_grams']
        ret_dur = ret_res['chosen_dur_min']

        # Υπολογισμός κυκλικής διαδρομής (Round Trip)
        student_co2 = go_co2 + ret_co2
        total_co2_grams += student_co2

        mode_counts[go_id] += 1
        mode_counts[ret_id] += 1
        mode_co2[go_id] += go_co2
        mode_co2[ret_id] += ret_co2

        print(f"Student {i:02d} (Postcode {s['tk']} | Grade {s['grade']}):\n"
              f"   Outbound (Peak): {go_name:<26} | CO2: {go_co2:>3} g | Time: {go_dur:>4} min\n"
              f"   Inbound (Off Peak): {ret_name:<26} | CO2: {ret_co2:>3} g | Time: {ret_dur:>4} min\n"
              f"   Round Trip Footprint: {student_co2} g CO2eq\n")

    print(f"\n============================================================")
    print(f"CO2 Environmental Footprint Summary (Grades {min_grade} to {max_grade}) in Round Trip")
    print(f"============================================================")
    print(f"Total students within grade range:            {len(filtered_students)}")
    print(f"Valid simulated student trips (Attica):       {valid_simulated}")
    print(f"Excluded students (noise or invalid postcode): {invalid_count}")
    print(f"============================================================")
    print(f"Total CO2 emissions (Round Trip):             {total_co2_grams:,.0f} g CO2eq ({total_co2_grams/1000:.2f} kg CO2)")
    if valid_simulated > 0:
        print(f"Average student footprint (Round Trip):       {total_co2_grams/valid_simulated:.1f} g CO2eq per student")
    print(f"============================================================")
    print(f"Mode Choice Distribution (Monte Carlo Round Trip Legs):")
    total_legs = valid_simulated * 2
    for m_id, name in [('transit1', 'Metro + Bus'), ('transit2', 'Direct Bus'), 
                      ('car', 'Car'), ('moto', 'Motorcycle'), ('foot', 'Walking')]:
        cnt = mode_counts[m_id]
        pct = (cnt / total_legs * 100) if total_legs > 0 else 0
        co2_sum = mode_co2[m_id]
        print(f"   * {name:<24}: {cnt:>2} legs ({pct:>4.1f}%) | CO2: {co2_sum:>6} g")
    print(f"============================================================")

## 📍 Section 7: Εκκίνηση της Προσομοίωσης

Τρέχουμε τη συνάρτηση προσομοίωσης για το εύρος βαθμών `0.0` έως `1.0`.

In [ ]:
run_simulation(0.0, 1.0)